## Prepare Env

In [ ]:
%pip install boto3 pyspark delta-spark

In [ ]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

## Ingestion
>### 1. Copy files from local to data lake layer files

In [ ]:
path_to_files = "/home/harry/Downloads/USA-20231226T104839Z-001/USA/Import (USA)/2020/US_Import_Mar_2020.csv"
bucket_name = "warehouse"
folder_target = "bol.file"
minio_access_key = 'demo-access-key'
minio_secret_key = 'demo-secret-key'
minio_endpoint = 'http://localhost:9000'  # Replace with your MinIO server endpoint

In [ ]:
# Create an S3 client with MinIO configuration
s3_client = boto3.client(
    's3',
    aws_access_key_id=minio_access_key,
    aws_secret_access_key=minio_secret_key,
    endpoint_url=minio_endpoint
)

In [ ]:
# Upload the file to the MinIO bucket
filename = path_to_files.split("/")[-1]
with open(path_to_files, 'rb') as data:
    s3_client.upload_fileobj(data, bucket_name, f"{folder_target}/{filename}")

print(f"File uploaded to MinIO bucket: {bucket_name}")

> ### 2. Write files to delta table

In [ ]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("JsonToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", f"http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "demo-access-key") \
    .config("spark.hadoop.fs.s3a.secret.key", "demo-secret-key") \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [ ]:
filename = path_to_files.split("/")[-1]
file_path = f"s3a://{bucket_name}/{folder_target}/{filename}"
table_name = f"{filename.split('.')[0]}.bronze.delta"

delta_table_path = f"s3a://{bucket_name}/{table_name}"

In [ ]:
# Read file into a DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

In [ ]:
# Lowercase and replace spaces with underscores for all column names
new_columns = [col(old_col).alias(old_col.lower().replace(' ', '_').replace('(', '|').replace(')', '|')) for old_col in df.columns]
df = df.select(*new_columns)

In [ ]:
# df.show()
df.columns

In [ ]:
# Write DataFrame to Delta table
df.write.format("delta").mode("overwrite").save(delta_table_path)

# Stop the Spark session
spark.stop()

# Processing